# Nightingale: synthetic patients from Synthea, and their patient knowledge graphs (task 1d), on Kaggle

This notebook runs **Synthea**, the open-source synthetic patient generator, at a **pinned release**
(v4.0.0) with **seed 42**, then builds each patient's knowledge graph with the repository's own code
(`scripts/build_patient_kg.py`, `src/patient_kg/`). Every patient is synthetic; nothing here touches
DDXPlus, the models or any real record. Nothing is installed on the laptop (D-7: Kaggle for every
cloud job).

What it writes, in one zip (`synthea_patient_kg.zip`):
- `csv/`: Synthea's CSV output (patients, conditions, observations, medications, procedures, …);
- `patient_kg/findings.jsonl`: each living patient's `RF:*` findings as of the reference date;
- `patient_kg/summary.json`: how many patients have each risk factor, and **every active condition
  code the SNOMED → RF table does not map**, with its reason, for review;
- `STATUS.txt` and `logs/`: the commit, Synthea's version and the jar's SHA-256, Java's version, the
  seed, the population and the reference date.

## How to run it (about 5 minutes of your time; the run itself about 10–30 minutes)

1. On kaggle.com: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. **Session options:** Accelerator **None** (Synthea uses the CPU only); **Internet on** (it clones
   the repository and downloads Synthea's release jar from GitHub).
3. **Save Version → Save & Run All (Commit)**. You can close the browser; the run continues.
4. When it is done, open the version's **Output**, download `synthea_patient_kg.zip`, and unzip it
   into `data/raw/synthea/` on the laptop. `STATUS.txt` says DONE or what went wrong.

The smoke test (cell 4) runs 5 patients first, so a broken setup stops within minutes. Java 17 or
newer is needed (Synthea v4's README); the setup cell installs it with apt if Kaggle's image lacks it.

In [ ]:
# The commit Claude gave you. "master" works too: STATUS.txt records the exact commit.
COMMIT = "master"
SYNTHEA_VERSION = "v4.0.0"  # pinned: src/patient_kg/mapping.py's table was checked against it
SEED = 42  # Synthea's -s and -cs (the population's and the clinicians' seeds)
POPULATION = 1000  # living patients Synthea generates
AGES = "25-90"  # adults, the ages a chest-pain presentation concerns
REFERENCE_DATE = "20260101"  # Synthea's -r: the day the population is simulated up to
STATE = "Massachusetts"  # Synthea's default state, named so the run is explicit

In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPO = Path("/tmp/nightingale")  # the code, outside /kaggle/working
WORK = Path("/tmp/synthea")  # the jar and the smoke test, outside /kaggle/working
OUT = Path("/kaggle/working/synthea_patient_kg")  # what you download
LOGS = OUT / "logs"
for folder in (WORK, LOGS):
    folder.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("commit", commit)
for module, package in (("pydantic", "pydantic"), ("networkx", "networkx"), ("yaml", "pyyaml")):
    try:
        __import__(module)
    except ImportError:
        run(sys.executable, "-m", "pip", "install", "-q", package)
sys.path.insert(0, str(REPO))
from src.patient_kg.mapping import SYNTHEA_VERSION as CHECKED

assert SYNTHEA_VERSION == CHECKED, f"the table was checked against {CHECKED}, not {SYNTHEA_VERSION}"


def java_major():
    try:
        text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return None, ""
    found = re.search(r'version "(\d+)', text)
    return (int(found.group(1)) if found else None), text.strip()


major, version_text = java_major()
if major is None or major < 17:
    print("Java", major, "is missing or too old for Synthea v4 (it needs 17+): installing OpenJDK 17")
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless")
    major, version_text = java_major()
assert major is not None and major >= 17, version_text
print(version_text)

In [ ]:
# Synthea's release jar, pinned by tag; its SHA-256 goes into STATUS.txt.
JAR = WORK / f"synthea-with-dependencies-{SYNTHEA_VERSION}.jar"
URL = f"https://github.com/synthetichealth/synthea/releases/download/{SYNTHEA_VERSION}/synthea-with-dependencies.jar"
if not JAR.exists():
    print("downloading", URL)
    urllib.request.urlretrieve(URL, JAR)
jar_sha256 = hashlib.sha256(JAR.read_bytes()).hexdigest()
print(JAR.name, f"{JAR.stat().st_size / 1e6:.0f} MB", "sha256", jar_sha256)


def synthea(population, base_directory, log):
    command = [
        "java", "-jar", JAR,
        "-s", SEED, "-cs", SEED, "-r", REFERENCE_DATE, "-p", population, "-a", AGES,
        f"--exporter.baseDirectory={base_directory}",
        "--exporter.csv.export=true",
        "--exporter.fhir.export=false",
        "--exporter.hospital.fhir.export=false",
        "--exporter.practitioner.fhir.export=false",
        STATE,
    ]
    print("$", " ".join(map(str, command)), flush=True)
    with open(log, "w", encoding="utf-8") as handle:
        return subprocess.run([str(c) for c in command], cwd=WORK, stdout=handle, stderr=subprocess.STDOUT).returncode


AS_OF = f"{REFERENCE_DATE[:4]}-{REFERENCE_DATE[4:6]}-{REFERENCE_DATE[6:]}"

In [ ]:
# Smoke test: 5 patients, then the repository's builder on them. A failure stops the run here.
smoke = WORK / "smoke"
shutil.rmtree(smoke, ignore_errors=True)
code = synthea(5, smoke, LOGS / "smoke.log")
assert code == 0, open(LOGS / "smoke.log", encoding="utf-8").read()[-3000:]
run(sys.executable, "scripts/build_patient_kg.py", "--synthea", smoke / "csv", "--as-of", AS_OF, "--out", WORK / "smoke_kg", cwd=REPO)

In [ ]:
# The real run: POPULATION patients, seed SEED. Synthea's log goes to logs/synthea.log.
started = time.time()
shutil.rmtree(OUT / "csv", ignore_errors=True)
code = synthea(POPULATION, OUT, LOGS / "synthea.log")
print(f"Synthea exit code {code} after {(time.time() - started) / 60:.0f} min")
print(open(LOGS / "synthea.log", encoding="utf-8").read()[-2000:])

In [ ]:
# The patient knowledge graphs, as of the reference date; the builder prints every active condition
# code the SNOMED -> RF table does not map. Then STATUS.txt, and the one zip to download.
lines = []
if (OUT / "csv" / "patients.csv").exists():
    with open(LOGS / "build.log", "w", encoding="utf-8") as handle:
        result = subprocess.run(
            [sys.executable, "scripts/build_patient_kg.py", "--synthea", str(OUT / "csv"),
             "--as-of", AS_OF, "--out", str(OUT / "patient_kg")],
            cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
        )
    print(open(LOGS / "build.log", encoding="utf-8").read())
    summary = json.loads((OUT / "patient_kg" / "summary.json").read_text(encoding="utf-8"))
    lines += [
        "DONE" if result.returncode == 0 else f"BUILD FAILED (exit {result.returncode}): see logs/build.log",
        f"commit {commit}",
        f"synthea {SYNTHEA_VERSION} sha256 {jar_sha256}",
        f"java {version_text.splitlines()[0]}",
        f"seed {SEED} population {POPULATION} ages {AGES} reference date {REFERENCE_DATE} state {STATE}",
        f"patients {summary['patients']}, alive on {summary['as_of']}: {summary['alive_on_as_of']}",
        f"concepts {json.dumps(summary['concepts'])}",
        f"unmapped active condition codes: {len(summary['unmapped_active_conditions'])}",
    ]
else:
    lines.append("NOT DONE: Synthea wrote no csv/patients.csv; see logs/synthea.log")
(OUT / "STATUS.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
archive = shutil.make_archive("/kaggle/working/synthea_patient_kg", "zip", OUT.parent, OUT.name)
print("download:", archive)